# Follow-up experiments

Four scripts, in order of value per hour spent.

| | What it answers | Needs | Time |
|---|---|---|---|
| 8 Shortcut test | Is the generator recoverable, and how much signal does it carry? | CPU | ~5 min |
| 9 Bootstrap difference | Is the fabrication-vs-other gap statistically real? | CPU | ~10 min |
| 10 Prompted-LLM detector | Does a prompted model also fail under source shift? | GPU | ~40 min |
| 11 NLI grounding | Does semantic grounding fix the omission gap? | GPU | ~2-3 h |

**8 and 9 are already in the paper** — run them to reproduce, or to regenerate
after any change to the labels. **10 and 11 are not**; they answer the two
experiments the paper currently lists as untried.

Set the runtime to GPU before Part 3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CANDIDATES = [
    '/content/drive/Shareddrives/RESEARCH/2026/PROMPTS/RESULTS',
    '/content/drive/Shareddrives/RESEARCH/Research/2026/PROMPTS/RESULTS',
]
ok = lambda p: os.path.isdir(os.path.join(p, 'EVALUATION-RESULT'))
RESULTS_ROOT = next((p for p in CANDIDATES if ok(p)), None)
if RESULTS_ROOT is None:
    raise SystemExit('Set RESULTS_ROOT by hand.')

OUT = f'{RESULTS_ROOT}/ANALYSIS-OUTPUT-RESULT/iclr_benchmark'
os.environ['HALLUBENCH_OUT'] = OUT
LABELS = f'{OUT}/benchmark_labels.csv.gz'
print('labels:', 'found' if os.path.exists(LABELS) else 'MISSING')

---
## Part 1 — Shortcut test (CPU, ~5 min)

Two measurements. First, how accurately a bag-of-words classifier identifies
which source LLM wrote a report. Second, how far a predictor that reads **no
report text at all** — returning only the writing model's base rate — gets
towards the full detector's AUC.

The second is the one that matters. If the base-rate predictor captures most of
the detector's above-chance signal on a label, then detection of that label is
mostly generator recognition.

In [ ]:
%%writefile /content/08_shortcut_test.py
#!/usr/bin/env python3
"""
Step 8 - does the detector read evidence, or recognise the writer?

Two measurements that test the shortcut directly rather than inferring it from
the transfer result:

  (a) SOURCE CLASSIFICATION. Train a classifier to predict which source LLM
      wrote a report. If this is easy, the shortcut is available.

  (b) BASE-RATE PREDICTOR. A predictor that sees no report text at all: it
      looks up which source LLM wrote the report and returns that source's
      training-set prevalence for the label. Whatever AUC this reaches is
      attributable to source identity alone. Comparing it against the full
      detector, on the above-chance scale, gives the share of the detector's
      signal that source identity supplies.

CPU only, a few minutes. No GPU, no network.

    python3 08_shortcut_test.py

Writes: shortcut_test.csv
"""
import argparse, os
import numpy as np, pandas as pd, warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
warnings.filterwarnings('ignore')

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']
AXIS = {'H1': 'fabrication', 'H5': 'fabrication', 'H6': 'fabrication',
        'H3': 'omission', 'H2': 'distortion', 'H4': 'distortion',
        'any_hallucination': 'binary'}

ap = argparse.ArgumentParser()
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/shortcut_test.csv')
args = ap.parse_args()

df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')

X = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=5,
                    sublinear_tf=True, strip_accents='unicode').fit_transform(df.model_output)
tr = (df.split_random == 'train').to_numpy()
te = (df.split_random == 'test').to_numpy()

# ---------------------------------------------------------------- (a)
print('=== (a) SOURCE-MODEL CLASSIFICATION (video-disjoint split) ===')
clf = LogisticRegression(max_iter=3000).fit(X[tr], df.model[tr])
pred = clf.predict(X[te])
acc = accuracy_score(df.model[te], pred)
mf1 = f1_score(df.model[te], pred, average='macro')
print(f'  accuracy  {acc:.3f}   (chance {1/df.model.nunique():.3f})')
print(f'  macro F1  {mf1:.3f}')
for m in sorted(df.model.unique()):
    sel = (df.model[te] == m).to_numpy()
    print(f'    {m:8s} recall {(pred[sel] == m).mean():.3f}')

# ---------------------------------------------------------------- (b)
print('\n=== (b) BASE-RATE PREDICTOR vs FULL DETECTOR ===')
print(f"{'type':6s}{'axis':13s}{'base rate':>10s}{'tfidf':>8s}{'share':>8s}")
rows = []
for t in TARGETS:
    y = df[t].to_numpy()
    if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
        continue
    # base-rate predictor: no text, just the source LLM's training prevalence
    rate = df.loc[tr].groupby('model')[t].mean()
    p_base = df.model[te].map(rate).to_numpy()
    auc_base = roc_auc_score(y[te], p_base)
    # full lexical detector
    d = LogisticRegression(max_iter=2000, class_weight='balanced',
                           random_state=42).fit(X[tr], y[tr])
    auc_full = roc_auc_score(y[te], d.predict_proba(X[te])[:, 1])
    share = (auc_base - 0.5) / (auc_full - 0.5) if auc_full > 0.5 else float('nan')
    rows.append({'target': t, 'axis': AXIS[t], 'auc_base_rate': round(auc_base, 3),
                 'auc_tfidf': round(auc_full, 3), 'share_of_signal': round(share, 3)})
    print(f'{t:6s}{AXIS[t]:13s}{auc_base:10.3f}{auc_full:8.3f}{share:7.0%}')

res = pd.DataFrame(rows)
res.attrs['source_accuracy'] = acc
res.to_csv(args.out, index=False)

fab = res[res.axis == 'fabrication'].share_of_signal
omi = res[res.axis == 'omission'].share_of_signal
print(f'\n  fabrication: {fab.min():.0%}-{fab.max():.0%} of the signal is source identity')
print(f'  omission   : {omi.mean():.0%}')
print('\n  -> that is why fabrication loses most when the source is withheld')
print('\nwrote ->', args.out)


In [ ]:
!python3 /content/08_shortcut_test.py --labels "$LABELS" --out "$OUT/shortcut_test.csv"

---
## Part 2 — Bootstrap the degradation difference (CPU, ~10 min)

A ratio of 1.4–3.1× is a description, not a test. This resamples both splits
1,000 times and reports a confidence interval on the difference itself.

Run it twice: once as-is, once with `--drop-h1` to confirm the result does not
depend on the low-reliability type.

In [ ]:
%%writefile /content/09_bootstrap_diff.py
#!/usr/bin/env python3
"""
Step 9 - is the degradation difference statistically real?

The paper reports fabrication degrading 1.4 to 3.1 times more than omission and
distortion. A ratio is not a test. This bootstraps the DIFFERENCE

    (fabrication LOSO - fabrication standard) - (other LOSO - other standard)

resampling both the standard-split test set and each leave-one-source-out fold,
and reports a 95% interval. If that interval excludes zero the asymmetry is not
an artifact of sampling.

CPU only, roughly 10 minutes for 1,000 resamples.

    python3 09_bootstrap_diff.py --n-boot 1000

Writes: bootstrap_diff.csv
"""
import argparse, os
import numpy as np, pandas as pd, warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
FAB = ['H1', 'H5', 'H6']
OTHER = ['H2', 'H3', 'H4']          # omission + distortion

ap = argparse.ArgumentParser()
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/bootstrap_diff.csv')
ap.add_argument('--n-boot', type=int, default=1000)
ap.add_argument('--drop-h1', action='store_true',
                help='exclude H1, the low-reliability type, from the fabrication axis')
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()

fab = [t for t in FAB if not (args.drop_h1 and t == 'H1')]
print('fabrication axis:', fab)

df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
X = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=5,
                    sublinear_tf=True, strip_accents='unicode').fit_transform(df.model_output)


def fit_predict(train_mask, test_mask):
    """Return {type: (y_true, y_score)} on the test set."""
    out = {}
    for t in H:
        y = df[t].to_numpy()
        if len(np.unique(y[train_mask])) < 2 or len(np.unique(y[test_mask])) < 2:
            continue
        c = LogisticRegression(max_iter=2000, class_weight='balanced',
                               random_state=42).fit(X[train_mask], y[train_mask])
        out[t] = (y[test_mask], c.predict_proba(X[test_mask])[:, 1])
    return out


print('fitting standard split ...', flush=True)
P_std = fit_predict((df.split_random == 'train').to_numpy(),
                    (df.split_random == 'test').to_numpy())
P_loso = {}
for held in sorted(df.model.unique()):
    print('fitting LOSO fold', held, flush=True)
    P_loso[held] = fit_predict((df.model != held).to_numpy(),
                               (df.model == held).to_numpy())


def axis_auc(P, idx, types):
    """Mean per-type AUC within an axis, on a bootstrap resample."""
    vals = []
    for t in types:
        if t not in P:
            return None
        y, p = P[t]
        i = idx[t]
        if len(np.unique(y[i])) < 2:
            return None
        vals.append(roc_auc_score(y[i], p[i]))
    return float(np.mean(vals))


rng = np.random.default_rng(args.seed)
diffs, fab_drops, oth_drops = [], [], []
for b in range(args.n_boot):
    idx_s = {t: rng.choice(len(P_std[t][0]), len(P_std[t][0]), replace=True) for t in P_std}
    f_s, o_s = axis_auc(P_std, idx_s, fab), axis_auc(P_std, idx_s, OTHER)
    if f_s is None or o_s is None:
        continue
    f_l, o_l = [], []
    for held in P_loso:
        idx_l = {t: rng.choice(len(P_loso[held][t][0]), len(P_loso[held][t][0]), replace=True)
                 for t in P_loso[held]}
        a = axis_auc(P_loso[held], idx_l, fab)
        c = axis_auc(P_loso[held], idx_l, OTHER)
        if a is None or c is None:
            break
        f_l.append(a); o_l.append(c)
    if len(f_l) < len(P_loso):
        continue
    fd = np.mean(f_l) - f_s
    od = np.mean(o_l) - o_s
    fab_drops.append(fd); oth_drops.append(od); diffs.append(fd - od)
    if (b + 1) % 200 == 0:
        print(f'  {b+1}/{args.n_boot}', flush=True)

d = np.array(diffs)
lo, hi = np.percentile(d, [2.5, 97.5])
print()
print('=== fabrication degradation minus omission/distortion degradation (TF-IDF) ===')
print(f'  fabrication drop      {np.mean(fab_drops):+.3f}')
print(f'  omission/distortion   {np.mean(oth_drops):+.3f}')
print(f'  difference            {d.mean():+.3f}')
print(f'  bootstrap 95%         [{lo:+.3f}, {hi:+.3f}]   ({len(d)} resamples)')
print(f'  excludes zero         {hi < 0}')

pd.DataFrame([{'axis_fabrication': '+'.join(fab),
               'fab_drop': round(float(np.mean(fab_drops)), 4),
               'other_drop': round(float(np.mean(oth_drops)), 4),
               'difference': round(float(d.mean()), 4),
               'ci_low': round(float(lo), 4), 'ci_high': round(float(hi), 4),
               'n_resamples': len(d), 'excludes_zero': bool(hi < 0)}]).to_csv(args.out, index=False)
print('\nwrote ->', args.out)


In [ ]:
!python3 /content/09_bootstrap_diff.py --labels "$LABELS" \
    --out "$OUT/bootstrap_diff.csv" --n-boot 1000

In [ ]:
!python3 /content/09_bootstrap_diff.py --labels "$LABELS" \
    --out "$OUT/bootstrap_diff_noH1.csv" --n-boot 1000 --drop-h1

---
## Part 3 — Prompted-LLM detector (GPU, ~40 min)

Runs an open-weight model locally. **No API key and no cost.**

The model must not be Claude, GPT or Gemini — prompting one of the corpus's own
writers reintroduces the self-preference bias the panel design exists to
exclude. The script aborts if the name looks like one of them.

Start at `--n-per-fold 100` and check the `parsed n/n` line before scaling. If
parsing fails, fix it there rather than after an hour of generation.

In [ ]:
!pip -q install "transformers>=4.48" accelerate
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

In [ ]:
%%writefile /content/10_llm_detector_local.py
#!/usr/bin/env python3
"""
Step 10 - prompted-LLM detector, run LOCALLY on the GPU. No API key, no cost.

Addresses the objection that the paper tests linear and encoder detectors but
not the form a practitioner would actually deploy: a frontier model prompted
zero-shot with the report and its reference.

The model MUST NOT be one of the three that wrote the corpus. Prompting Claude,
GPT or Gemini here would reintroduce the self-preference bias the panel design
exists to exclude, and the number would be uninterpretable. Qwen and Llama are
safe choices and run under `transformers` with no API.

Zero-shot, so there is no training split: the held-out-source condition is
simply the reports from that source LLM.

    pip install "transformers>=4.48" accelerate bitsandbytes
    python3 10_llm_detector_local.py --model Qwen/Qwen2.5-7B-Instruct --n-per-fold 300

Writes: llm_detector_results.csv, llm_detector_raw.csv

RUNTIME. Roughly 1.5 s per report on an A100 in bfloat16. At --n-per-fold 300
over five conditions that is about 40 minutes; at 1,000 it is over two hours.
Start at 100 to check the prompt and the JSON parsing before scaling.
"""
import argparse, json, os, re
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']

RUBRIC = """You are auditing a forensic video report against an expert reference annotation of the same video. For each category decide whether the report contains that error.

H1 scene fabrication: describes a setting or event with no source in the footage
H2 crime misclassification: reports the act as a different category of crime
H3 crime missed: fails to report the criminal act under investigation
H4 severity minimization: understates the gravity of the act
H5 entity fabrication: introduces objects or details not present
H6 phantom actors: introduces people who do not appear

REFERENCE ANNOTATION:
{ref}

REPORT:
{rep}

Answer with JSON only, no prose:
{{"H1":0,"H2":0,"H3":0,"H4":0,"H5":0,"H6":0}}"""

ap = argparse.ArgumentParser()
ap.add_argument('--model', default='Qwen/Qwen2.5-7B-Instruct',
                help='must NOT be Claude, GPT or Gemini')
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/llm_detector_results.csv')
ap.add_argument('--raw', default=os.environ.get('HALLUBENCH_OUT', '.') + '/llm_detector_raw.csv')
ap.add_argument('--n-per-fold', type=int, default=300)
ap.add_argument('--batch', type=int, default=8)
ap.add_argument('--max-ref', type=int, default=1500)
ap.add_argument('--max-rep', type=int, default=3000)
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()

for banned in ('claude', 'gpt', 'gemini'):
    if banned in args.model.lower():
        raise SystemExit(f'ABORT: {args.model} looks like a source LLM of this corpus. '
                         'Using one as the detector reintroduces self-preference bias.')

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '| model:', args.model)
tok = AutoTokenizer.from_pretrained(args.model, padding_side='left')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    args.model, torch_dtype=torch.bfloat16 if dev == 'cuda' else torch.float32,
    device_map='auto')
model.eval()


def parse(txt):
    """Pull the six labels out of the reply. None if unparseable."""
    m = re.search(r'\{[^{}]*\}', txt or '', re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
    except Exception:
        return None
    if not all(h in d for h in H):
        return None
    try:
        return {h: int(bool(int(d[h]))) for h in H}
    except Exception:
        return None


def score(frame):
    """Score one test set, batched."""
    prompts = [tok.apply_chat_template(
        [{'role': 'user', 'content': RUBRIC.format(
            ref=str(r.ground_truth)[:args.max_ref],
            rep=str(r.model_output)[:args.max_rep])}],
        tokenize=False, add_generation_prompt=True) for _, r in frame.iterrows()]
    out = {}
    for s in range(0, len(prompts), args.batch):
        chunk = prompts[s:s + args.batch]
        enc = tok(chunk, return_tensors='pt', padding=True,
                  truncation=True, max_length=4096).to(dev)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=64, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        for k, g in enumerate(gen):
            reply = tok.decode(g[enc['input_ids'].shape[1]:], skip_special_tokens=True)
            out[frame.index[s + k]] = parse(reply)
        if s % (args.batch * 10) == 0:
            print(f'   {s}/{len(prompts)}', flush=True)
    return out


df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
df['ground_truth'] = df.ground_truth.fillna('')


def subsample(frame):
    if len(frame) <= args.n_per_fold:
        return frame
    per = max(1, args.n_per_fold // frame.crime_type.nunique())
    return (frame.groupby('crime_type', group_keys=False)
                 .apply(lambda g: g.sample(min(len(g), per), random_state=args.seed)))


conditions = {'random': df[df.split_random == 'test'],
              'heldout_technique': df[df.split_heldout_technique == 'test']}
for held in sorted(df.model.unique()):
    conditions[f'heldout_model_{held}'] = df[df.model == held]

rows, raw = [], []
for tag, frame in conditions.items():
    sub = subsample(frame).copy()
    print(f'\n== {tag}: scoring {len(sub):,} reports', flush=True)
    verdicts = score(sub)
    ok = [i for i, v in verdicts.items() if v is not None]
    print(f'   parsed {len(ok)}/{len(sub)}')
    if not ok:
        print('   NOTHING PARSED - check the prompt or the model before scaling up')
        continue
    sub = sub.loc[ok]
    P = pd.DataFrame([verdicts[i] for i in ok], index=ok)
    P['any_hallucination'] = P[H].max(axis=1)
    raw.append(pd.DataFrame({'split': tag, 'video': sub.video.values,
                             'model': sub.model.values, 'technique': sub.technique.values,
                             **{f'pred_{h}': P[h].values for h in TARGETS},
                             **{f'gold_{h}': sub[h].values for h in TARGETS}}))
    for t in TARGETS:
        y, p = sub[t].to_numpy(), P[t].to_numpy()
        if len(np.unique(y)) < 2:
            continue
        rows.append({'features': 'llm_detector', 'split': tag, 'target': t,
                     'auc': roc_auc_score(y, p), 'f1': f1_score(y, p),
                     'pos_rate_test': float(y.mean()), 'n': len(y)})

res = pd.DataFrame(rows)
res.to_csv(args.out, index=False)
if raw:
    pd.concat(raw).to_csv(args.raw, index=False)
print('\n' + res.pivot_table(index='split', columns='target', values='auc')[TARGETS].round(3).to_markdown())
print('\nNOTE: this detector emits hard 0/1 labels, so its AUC is computed on binary')
print('predictions and is NOT directly comparable to the probabilistic baselines.')
print('Report F1 alongside it and say so in the caption.')
print('wrote ->', args.out)


In [ ]:
!python3 /content/10_llm_detector_local.py --model Qwen/Qwen2.5-7B-Instruct \
    --n-per-fold 100 --labels "$LABELS" \
    --out "$OUT/llm_detector_smoke.csv" --raw "$OUT/llm_detector_smoke_raw.csv"

In [ ]:
!python3 /content/10_llm_detector_local.py --model Qwen/Qwen2.5-7B-Instruct \
    --n-per-fold 300 --labels "$LABELS" \
    --out "$OUT/llm_detector_results.csv" --raw "$OUT/llm_detector_raw.csv"

---
## Part 4 — NLI grounding (GPU, ~2-3 h)

The paper's grounded detector is lexical and fails on omission (0.577 under
shift against TF-IDF's 0.793). This scores every report sentence against the
reference with a cross-encoder NLI model, in **both** directions: report
sentences the reference does not support (fabrication), and reference sentences
the report does not carry (omission).

The reversed direction is the point. Lexical coverage failed on omission; if
entailment in that direction also fails, the omission gap is not a lexical
problem at all, which is itself worth reporting.

Features cache to `nli_features.parquet`, so `--reuse-feats` re-runs the
classifiers in seconds.

In [ ]:
%%writefile /content/11_nli_grounding.py
#!/usr/bin/env python3
"""
Step 11 - entailment-based grounded features.

The paper's `grounded` detector is lexical: token overlap, Jaccard, TF-IDF
cosine. It recovers binary detection under shift but fails on omission, and the
paper says a semantic version is the most promising direction not tried. This
is that version.

For each report, every sentence is scored against the reference annotation by a
cross-encoder NLI model, giving per-sentence entailment / neutral /
contradiction probabilities. Those are aggregated into report-level features:

  FABRICATION side - report sentences the reference does not support
    mean/max contradiction of report sentences given the reference
    fraction of report sentences with entailment below a threshold
  OMISSION side - reference content the report does not carry
    the same, with the direction reversed (reference sentences as hypotheses)

The reversed direction is the point: lexical coverage failed on omission, and
entailment in that direction is what should capture "the report drops the
criminal act".

Features then feed the same logistic regression as every other baseline, so the
comparison is like-for-like.

    pip install "transformers>=4.48" accelerate
    python3 11_nli_grounding.py --nli cross-encoder/nli-deberta-v3-base

Writes: nli_features.parquet, nli_results.csv

RUNTIME. 19,361 reports at ~20 sentences each, both directions, is ~800k pairs.
On an A100 in bfloat16 at batch 128 that is roughly 2-3 hours. --max-sent caps
sentences per report; 15 keeps it near 2 hours with little loss.
"""
import argparse, os, re
import numpy as np, pandas as pd, torch, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
warnings.filterwarnings('ignore')

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']

ap = argparse.ArgumentParser()
ap.add_argument('--nli', default='cross-encoder/nli-deberta-v3-base')
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--feats', default=os.environ.get('HALLUBENCH_OUT', '.') + '/nli_features.parquet')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/nli_results.csv')
ap.add_argument('--max-sent', type=int, default=15, help='sentences per document')
ap.add_argument('--batch', type=int, default=128)
ap.add_argument('--reuse-feats', action='store_true', help='skip scoring, load cached features')
args = ap.parse_args()

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
df['ground_truth'] = df.ground_truth.fillna('')

SENT = re.compile(r'(?<=[.!?])\s+')
def sents(t, cap):
    s = [x.strip() for x in SENT.split(str(t)) if len(x.strip()) > 15]
    return s[:cap] if s else ['']

if args.reuse_feats and os.path.exists(args.feats):
    F = pd.read_parquet(args.feats)
    print('loaded cached features', F.shape)
else:
    print('device:', dev, '| NLI model:', args.nli)
    tok = AutoTokenizer.from_pretrained(args.nli)
    nli = AutoModelForSequenceClassification.from_pretrained(
        args.nli, torch_dtype=torch.bfloat16 if dev == 'cuda' else torch.float32).to(dev).eval()
    # label order differs between checkpoints; read it off the config
    id2 = {i: l.lower() for i, l in nli.config.id2label.items()}
    ENT = [i for i, l in id2.items() if 'entail' in l][0]
    CON = [i for i, l in id2.items() if 'contra' in l][0]
    print('  entailment index', ENT, '| contradiction index', CON)

    def score_pairs(premises, hypotheses):
        out = []
        for s in range(0, len(premises), args.batch):
            enc = tok(premises[s:s + args.batch], hypotheses[s:s + args.batch],
                      return_tensors='pt', padding=True, truncation=True,
                      max_length=256).to(dev)
            with torch.no_grad():
                p = torch.softmax(nli(**enc).logits.float(), dim=-1).cpu().numpy()
            out.append(p)
        return np.vstack(out) if out else np.zeros((0, 3))

    rows = []
    for n, (_, r) in enumerate(df.iterrows()):
        rep_s = sents(r.model_output, args.max_sent)
        ref_s = sents(r.ground_truth, args.max_sent)
        ref_joined = ' '.join(ref_s)[:2000]
        rep_joined = ' '.join(rep_s)[:2000]
        # direction 1: does the reference support each report sentence? (fabrication)
        f = score_pairs([ref_joined] * len(rep_s), rep_s)
        # direction 2: does the report carry each reference sentence? (omission)
        o = score_pairs([rep_joined] * len(ref_s), ref_s)
        rows.append({
            'fab_contra_mean': float(f[:, CON].mean()), 'fab_contra_max': float(f[:, CON].max()),
            'fab_entail_mean': float(f[:, ENT].mean()),
            'fab_unsupported_frac': float((f[:, ENT] < 0.5).mean()),
            'omi_entail_mean': float(o[:, ENT].mean()), 'omi_entail_min': float(o[:, ENT].min()),
            'omi_contra_mean': float(o[:, CON].mean()),
            'omi_dropped_frac': float((o[:, ENT] < 0.5).mean()),
            'n_rep_sent': len(rep_s), 'n_ref_sent': len(ref_s)})
        if (n + 1) % 500 == 0:
            print(f'  {n+1}/{len(df)}', flush=True)
    F = pd.DataFrame(rows)
    F.to_parquet(args.feats)
    print('wrote features ->', args.feats)

Xn = StandardScaler().fit_transform(F.to_numpy(dtype=float))

def run(train_mask, test_mask, tag):
    out = []
    for t in TARGETS:
        y = df[t].to_numpy()
        if len(np.unique(y[train_mask])) < 2 or len(np.unique(y[test_mask])) < 2:
            continue
        c = LogisticRegression(max_iter=3000, class_weight='balanced',
                               random_state=42).fit(Xn[train_mask], y[train_mask])
        out.append({'features': 'nli_grounded', 'split': tag, 'target': t,
                    'auc': roc_auc_score(y[test_mask], c.predict_proba(Xn[test_mask])[:, 1])})
    return out

res = []
res += run((df.split_random == 'train').to_numpy(), (df.split_random == 'test').to_numpy(), 'random')
for held in sorted(df.model.unique()):
    res += run((df.model != held).to_numpy(), (df.model == held).to_numpy(), f'heldout_model_{held}')
res += run((df.split_heldout_technique == 'train').to_numpy(),
           (df.split_heldout_technique == 'test').to_numpy(), 'heldout_technique')

r = pd.DataFrame(res); r.to_csv(args.out, index=False)
print('\n' + r.pivot_table(index='split', columns='target', values='auc')[TARGETS].round(3).to_markdown())
print('\nThe comparison that matters is omission (H3) under leave-one-source-out:')
print('lexical grounding reaches 0.577 there. If entailment beats that, semantic')
print('grounding is the answer to the omission gap; if not, the gap is not lexical.')
print('wrote ->', args.out)


In [ ]:
!python3 /content/11_nli_grounding.py --nli cross-encoder/nli-deberta-v3-base \
    --max-sent 15 --batch 128 --labels "$LABELS" \
    --feats "$OUT/nli_features.parquet" --out "$OUT/nli_results.csv"

### Send back

- `shortcut_test.csv`, `bootstrap_diff.csv`, `bootstrap_diff_noH1.csv` — these
  reproduce what is already in the paper; flag any disagreement
- `llm_detector_results.csv` — new; would close the most-requested gap
- `nli_results.csv` — new; the omission row under `heldout_model_*` is the one
  to look at first